# v1.3 Dataset Builder: User + Test + Cohort

Build the v1.3 Test / Exercise Readiness dataset from database tables without storing credentials in the notebook.

This is not the full Learn Smarter model. It exports attempt-level DQ flags, a published KPI dataset, and a proxy-sequence dataset for BLS/ALS/CAS proxy analysis.

## v1.3 Boundaries

- Topic-level readiness is deferred to v2 because the current source has no canonical `topic_id` / `topic_name`.
- Subject context may be inferred cautiously from `tests.name`, but it is not canonical metadata.
- `class_id` can support cohort analysis after membership validation.
- BLS, ALS, CAS, and learning gain are proxy signals only in v1.3.
- DQ must annotate before filtering so the audit trail is preserved.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

try:
    from sqlalchemy import create_engine, inspect, text
except ImportError:
    create_engine = None
    inspect = None
    text = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
EXPORT_DIR = PROJECT_ROOT / "data" / "v1_3_dataset_build"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = {
    "test_takers": "test_takers",
    "test_results": "test_results",
    "test_questions": "test_questions",
    "test_answers": "test_answers",
    "tests": "tests",
    "users": "users",
    "test_classes": "test_classes",
    "class_memberships": None,
}

DATE_START = None
DATE_END = None
RUN_LABEL = "v1_3_user_test_cohort_build"

def build_engine():
    """Fill this in locally. Do not commit credentials."""
    if create_engine is None:
        raise ImportError("Install sqlalchemy and the database driver for your connection.")
    # import os
    # return create_engine(os.environ["ECAMPUS_DB_URL"])
    # from google.colab import userdata
    # return create_engine(userdata.get("ECAMPUS_DB_URL"))
    raise NotImplementedError("Add private DB connection details in your local copy.")

def inspect_tables(engine, table_names):
    db = inspect(engine)
    rows = []
    for logical_name, table_name in table_names.items():
        if not table_name:
            rows.append({"logical_name": logical_name, "table_name": None, "column_name": None, "type": None})
            continue
        try:
            columns = db.get_columns(table_name)
        except Exception as exc:
            rows.append({"logical_name": logical_name, "table_name": table_name, "column_name": "<inspect_failed>", "type": str(exc)})
            continue
        for column in columns:
            rows.append({"logical_name": logical_name, "table_name": table_name, "column_name": column.get("name"), "type": str(column.get("type"))})
    return pd.DataFrame(rows)

# engine = build_engine()
# schema_df = inspect_tables(engine, TABLES)
# display(schema_df)
# schema_df.to_csv(EXPORT_DIR / "schema_inventory.csv", index=False)

## Raw Extraction Query

`marks` comes from `test_takers.marks` because answer/result grade sums can include both correct and wrong answer rows.

Important denominator rule: `COUNT(test_questions)` is `question_bank_count` / `max_marks_db` context for randomized pools. It is not the normal accuracy denominator.

In [ ]:
RAW_ATTEMPT_SQL = """
WITH result_dedup AS (
    SELECT
        tr.*,
        ROW_NUMBER() OVER (
            PARTITION BY tr.test_taker_id, tr.test_question_id
            ORDER BY tr.updated_at DESC
        ) AS result_rank,
        COUNT(*) OVER (PARTITION BY tr.test_taker_id, tr.test_question_id) AS result_pair_count
    FROM {test_results} tr
), answer_rollup AS (
    SELECT
        rd.test_taker_id,
        COUNT(*) AS answer_rows,
        COUNT(DISTINCT rd.test_question_id) AS delivered_result_questions,
        COUNT(rd.chosen_answer_id) AS attempted_questions,
        SUM(CASE WHEN ta.correct = 1 THEN 1 ELSE 0 END) AS correct_answers,
        SUM(CASE WHEN rd.chosen_answer_id IS NOT NULL AND COALESCE(ta.correct, 0) = 0 THEN 1 ELSE 0 END) AS wrong_answers,
        SUM(COALESCE(rd.grade, 0)) AS answer_grade_sum_diagnostic,
        SUM(CASE WHEN rd.result_pair_count > 1 THEN rd.result_pair_count - 1 ELSE 0 END) AS duplicate_attempt_question_rows
    FROM result_dedup rd
    LEFT JOIN {test_answers} ta ON ta.answer_id = rd.chosen_answer_id
    WHERE rd.result_rank = 1
    GROUP BY rd.test_taker_id
), question_rollup AS (
    SELECT
        tq.test_id,
        COUNT(*) AS question_bank_count,
        SUM(COALESCE(tq.grade, 1)) AS total_marks_db
    FROM {test_questions} tq
    WHERE COALESCE(tq.is_active, 1) = 1
    GROUP BY tq.test_id
), test_class_rollup AS (
    SELECT
        tc.test_id,
        MIN(tc.class_id) AS class_id,
        MIN(tc.group_id) AS group_id,
        COUNT(DISTINCT tc.class_id) AS test_class_count
    FROM {test_classes} tc
    WHERE COALESCE(tc.active, 1) = 1
    GROUP BY tc.test_id
)
SELECT
    tt.test_taker_id AS attempt_id,
    tt.test_taker_id,
    tt.user_id,
    tt.subscriber_id,
    u.f_name AS first_name,
    u.l_name AS last_name,
    u.username AS learner_id,
    u.country,
    u.institute,
    u.city,
    tt.test_id,
    t.name AS test_name,
    tt.created_at,
    tt.finished_at,
    t.duration AS time_limit,
    t.pass_mark,
    t.question_limit,
    tt.no_of_questions,
    ar.answer_rows,
    ar.delivered_result_questions,
    ar.duplicate_attempt_question_rows,
    ar.attempted_questions,
    ar.correct_answers,
    ar.wrong_answers,
    tt.marks AS marks,
    ar.answer_grade_sum_diagnostic,
    qr.question_bank_count,
    qr.question_bank_count AS max_marks_db,
    qr.question_bank_count AS total_questions,
    qr.total_marks_db,
    qr.total_marks_db AS total_marks,
    tcr.class_id,
    tcr.group_id,
    tcr.test_class_count
FROM {test_takers} tt
LEFT JOIN {tests} t ON t.test_id = tt.test_id
LEFT JOIN {users} u ON u.user_id = tt.user_id
LEFT JOIN answer_rollup ar ON ar.test_taker_id = tt.test_taker_id
LEFT JOIN question_rollup qr ON qr.test_id = tt.test_id
LEFT JOIN test_class_rollup tcr ON tcr.test_id = tt.test_id
WHERE (:date_start IS NULL OR tt.created_at >= :date_start)
  AND (:date_end IS NULL OR tt.created_at < :date_end)
""".strip()

def render_sql(sql_template, tables):
    required = sorted(set(re.findall(r"\{([a-zA-Z0-9_]+)\}", sql_template)))
    missing = [name for name in required if not tables.get(name)]
    if missing:
        raise ValueError(f"Missing table mapping for: {missing}")
    return sql_template.format(**tables)

def read_raw_attempts(engine):
    sql = render_sql(RAW_ATTEMPT_SQL, TABLES)
    params = {"date_start": DATE_START, "date_end": DATE_END}
    return pd.read_sql_query(text(sql), engine, params=params)

# raw_attempts = read_raw_attempts(engine)
# display(raw_attempts.head())

## DQ Annotation And Delivered Denominator

Denominator priority:
1. `delivered_denominator` if already present
2. `delivered_result_questions`
3. `answer_grade_sum_diagnostic`
4. consistent `no_of_questions`
5. consistent `question_limit`
6. low-confidence legacy fallback only if no delivered evidence exists

`max_marks_db` / `question_bank_count` is flagged as bank-size context when it is materially larger than delivered evidence.

In [ ]:
def numeric_col(df, column_name):
    if column_name not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[column_name], errors="coerce")

def valid_denominator(candidate, df):
    marks = numeric_col(df, "marks")
    correct = numeric_col(df, "correct_answers")
    attempted = numeric_col(df, "attempted_questions_raw").fillna(numeric_col(df, "attempted_questions"))
    valid = candidate.notna() & candidate.gt(0)
    valid &= marks.isna() | marks.le(candidate)
    valid &= correct.isna() | correct.le(candidate)
    valid &= attempted.isna() | attempted.le(candidate)
    return valid.fillna(False)

def bank_like(candidate, df):
    anchors = pd.Series(np.nan, index=df.index, dtype="float64")
    for col in ["delivered_result_questions", "answer_grade_sum_diagnostic", "no_of_questions", "question_limit"]:
        value = numeric_col(df, col)
        anchors = anchors.fillna(value.where(valid_denominator(value, df)))
    return (candidate.notna() & anchors.notna() & candidate.gt(anchors + 5) & candidate.gt(anchors * 1.25)).fillna(False)

def canonical_denominator_with_source(df):
    denom = pd.Series(np.nan, index=df.index, dtype="float64")
    source = pd.Series("missing", index=df.index, dtype="object")

    for col in ["delivered_denominator", "delivered_result_questions", "answer_grade_sum_diagnostic", "no_of_questions", "question_limit"]:
        candidate = numeric_col(df, col)
        use = denom.isna() & valid_denominator(candidate, df)
        denom = denom.where(~use, candidate)
        source = source.where(~use, col)

    for col in ["max_marks_effective", "accuracy_denominator", "total_questions", "max_marks_db", "question_bank_count"]:
        candidate = numeric_col(df, col)
        use = denom.isna() & valid_denominator(candidate, df) & ~bank_like(candidate, df)
        denom = denom.where(~use, candidate)
        source = source.where(~use, col)

    for col in ["max_marks_db", "question_bank_count", "total_questions"]:
        candidate = numeric_col(df, col)
        use = denom.isna() & valid_denominator(candidate, df)
        denom = denom.where(~use, candidate)
        source = source.where(~use, f"{col}_last_resort")

    no_of_questions = numeric_col(df, "no_of_questions")
    question_limit = numeric_col(df, "question_limit")
    answer_sum = numeric_col(df, "answer_grade_sum_diagnostic")
    result_questions = numeric_col(df, "delivered_result_questions")
    max_marks_db = numeric_col(df, "max_marks_db")
    total_marks = numeric_col(df, "total_marks")
    total_questions = numeric_col(df, "total_questions")

    no_of_questions_consistent = valid_denominator(no_of_questions, df)
    no_of_questions_consistent &= answer_sum.isna() | answer_sum.le(0) | no_of_questions.sub(answer_sum).abs().le(1)
    no_of_questions_consistent &= result_questions.isna() | result_questions.le(0) | no_of_questions.sub(result_questions).abs().le(1)
    no_of_questions_consistent &= question_limit.isna() | question_limit.le(0) | no_of_questions.sub(question_limit).abs().le(1)

    question_limit_consistent = valid_denominator(question_limit, df)
    question_limit_consistent &= answer_sum.isna() | answer_sum.le(0) | question_limit.sub(answer_sum).abs().le(1)
    question_limit_consistent &= result_questions.isna() | result_questions.le(0) | question_limit.sub(result_questions).abs().le(1)
    question_limit_consistent &= no_of_questions.isna() | no_of_questions.le(0) | question_limit.sub(no_of_questions).abs().le(1)

    max_marks_db_is_bank_count = bank_like(max_marks_db, df)
    total_marks_suspect = total_marks.gt(0) & denom.gt(0) & total_marks.gt(denom * 1.25) & total_marks.gt(denom + 5)
    total_questions_suspect = total_questions.gt(0) & denom.gt(0) & total_questions.ne(denom) & bank_like(total_questions, df)
    denominator_conflict_flag = (denom.isna() | (no_of_questions.notna() & no_of_questions.gt(0) & ~no_of_questions_consistent) | (question_limit.notna() & question_limit.gt(0) & ~question_limit_consistent) | max_marks_db_is_bank_count).fillna(False)

    confidence = pd.Series("low", index=df.index, dtype="object")
    confidence = confidence.where(~source.isin(["delivered_denominator", "delivered_result_questions", "answer_grade_sum_diagnostic", "no_of_questions"]), "high")
    confidence = confidence.where(~source.eq("question_limit"), "medium")
    confidence = confidence.where(~source.str.endswith("_last_resort", na=False), "low")
    confidence = confidence.where(~denom.isna(), "missing")

    return denom.where(denom.gt(0)), source, confidence, max_marks_db_is_bank_count.fillna(False), no_of_questions_consistent.fillna(False), question_limit_consistent.fillna(False), denominator_conflict_flag, total_marks_suspect.fillna(False), total_questions_suspect.fillna(False)

def add_dq_flags(raw_df):
    df = raw_df.copy()
    for col in ["created_at", "finished_at"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df["attempted_questions_raw"] = numeric_col(df, "attempted_questions")
    df["correct_answers"] = numeric_col(df, "correct_answers")
    df["wrong_answers"] = numeric_col(df, "wrong_answers")
    df["marks"] = numeric_col(df, "marks")
    (df["accuracy_denominator"], df["accuracy_denominator_source"], df["denominator_confidence"], df["max_marks_db_is_bank_count"], df["no_of_questions_consistent"], df["question_limit_consistent"], df["denominator_conflict_flag"], df["total_marks_suspect"], df["total_questions_suspect"]) = canonical_denominator_with_source(df)
    df["delivered_denominator"] = df["accuracy_denominator"]
    df["delivered_denominator_source"] = df["accuracy_denominator_source"]
    df["full_test_accuracy"] = (df["marks"] / df["accuracy_denominator"]).clip(lower=0, upper=1)
    df["attempted_accuracy"] = (df["correct_answers"] / df["attempted_questions_raw"]).replace([np.inf, -np.inf], np.nan).clip(lower=0, upper=1)
    df["avg_accuracy_safe"] = df["full_test_accuracy"]

    no_of_questions = numeric_col(df, "no_of_questions")
    df["no_of_questions_suspect"] = (no_of_questions.isna() | no_of_questions.le(0) | (df["marks"].notna() & no_of_questions.notna() & df["marks"].gt(no_of_questions)) | (df["correct_answers"].notna() & no_of_questions.notna() & df["correct_answers"].gt(no_of_questions)) | (df["attempted_questions_raw"].notna() & no_of_questions.notna() & df["attempted_questions_raw"].gt(no_of_questions)) | (no_of_questions.notna() & no_of_questions.gt(0) & ~df["no_of_questions_consistent"])).fillna(True)

    finished_exists = "finished_at" in raw_df.columns
    activity_evidence = df["attempted_questions_raw"].fillna(0).gt(0) | df["correct_answers"].fillna(0).gt(0) | df["wrong_answers"].fillna(0).gt(0) | df["marks"].fillna(0).gt(0)
    if finished_exists:
        df["missing_finished_at_column"] = False
        df["completion_source"] = "finished_at"
        df["completion_status"] = np.where(df["finished_at"].notna(), "verified_complete", "incomplete")
    else:
        df["missing_finished_at_column"] = True
        df["completion_source"] = np.where(activity_evidence, "fallback_activity_evidence", "missing_finished_at_column")
        df["completion_status"] = np.where(activity_evidence, "unknown_but_usable", "incomplete")

    df["inactive_zero_attempt"] = df["attempted_questions_raw"].fillna(0).eq(0)
    df["missing_question_support"] = df["accuracy_denominator"].isna()
    df["missing_pass_mark"] = numeric_col(df, "pass_mark").isna()
    df["exact_duplicate_row"] = df.duplicated(keep=False)
    df["dq_eligible_proxy_sequence"] = df["completion_status"].isin(["verified_complete", "unknown_but_usable"]) & df["accuracy_denominator"].notna() & ~df["inactive_zero_attempt"]
    df["dq_eligible_published"] = df["dq_eligible_proxy_sequence"] & ~df["exact_duplicate_row"]
    return df

# dq_attempts = add_dq_flags(raw_attempts)
# display(dq_attempts.head())

## Output Builders

Published KPI data may dedupe best attempt per learner/test. Proxy-sequence data must preserve repeated attempts for BLS/ALS/CAS proxy logic.

In [ ]:
def build_published_kpi_dataset(dq_attempts):
    df = dq_attempts.loc[dq_attempts["dq_eligible_published"]].copy()
    sort_cols = [col for col in ["user_id", "test_id", "avg_accuracy_safe", "marks", "created_at"] if col in df.columns]
    ascending = [True, True, False, False, False][:len(sort_cols)]
    df = df.sort_values(sort_cols, ascending=ascending)
    return df.drop_duplicates(["user_id", "test_id"], keep="first")

def build_proxy_sequence_dataset(dq_attempts):
    df = dq_attempts.loc[dq_attempts["dq_eligible_proxy_sequence"]].copy()
    return df.sort_values(["user_id", "test_id", "created_at", "attempt_id"], na_position="last")

def build_user_test_readiness_summary(proxy_sequence, published_kpi=None):
    rows = []
    for (user_id, test_id), group in proxy_sequence.groupby(["user_id", "test_id"], dropna=False):
        attempts = group.sort_values(["created_at", "attempt_id"], na_position="last")
        first = attempts.iloc[0]
        later = attempts.iloc[1:]
        current = later.iloc[-1] if len(later) else pd.Series(dtype="object")
        potential = later.sort_values("avg_accuracy_safe", ascending=False).iloc[0] if len(later) else pd.Series(dtype="object")
        bls = first.get("avg_accuracy_safe", np.nan)
        current_als = current.get("avg_accuracy_safe", np.nan) if len(later) else np.nan
        potential_als = potential.get("avg_accuracy_safe", np.nan) if len(later) else np.nan
        rows.append({
            "user_id": user_id, "test_id": test_id, "test_name": first.get("test_name"), "subscriber_id": first.get("subscriber_id"),
            "first_name": first.get("first_name"), "last_name": first.get("last_name"), "learner_id": first.get("learner_id"),
            "country": first.get("country"), "institute": first.get("institute"), "city": first.get("city"), "class_id": first.get("class_id"), "group_id": first.get("group_id"), "attempt_count": len(attempts),
            "bls_attempt_id": first.get("attempt_id"), "bls_created_at": first.get("created_at"), "bls_score_raw": first.get("marks"), "bls_score_denominator": first.get("accuracy_denominator"), "bls_score_denominator_source": first.get("accuracy_denominator_source"), "bls_score_pct": bls * 100 if pd.notna(bls) else np.nan,
            "current_als_attempt_id": current.get("attempt_id") if len(later) else np.nan, "current_als_created_at": current.get("created_at") if len(later) else pd.NaT, "current_als_score_raw": current.get("marks") if len(later) else np.nan, "current_als_score_denominator": current.get("accuracy_denominator") if len(later) else np.nan, "current_als_score_denominator_source": current.get("accuracy_denominator_source") if len(later) else np.nan, "current_als_score_pct": current_als * 100 if pd.notna(current_als) else np.nan,
            "potential_als_attempt_id": potential.get("attempt_id") if len(later) else np.nan, "potential_als_created_at": potential.get("created_at") if len(later) else pd.NaT, "potential_als_score_raw": potential.get("marks") if len(later) else np.nan, "potential_als_score_denominator": potential.get("accuracy_denominator") if len(later) else np.nan, "potential_als_score_denominator_source": potential.get("accuracy_denominator_source") if len(later) else np.nan, "potential_als_score_pct": potential_als * 100 if pd.notna(potential_als) else np.nan,
            "learning_gain_pct": (current_als - bls) * 100 if pd.notna(current_als) and pd.notna(bls) else np.nan, "potential_gain_pct": (potential_als - bls) * 100 if pd.notna(potential_als) and pd.notna(bls) else np.nan,
            "proxy_evidence_band": "high" if len(attempts) >= 3 else ("medium" if len(attempts) == 2 else "low"), "learn_smarter_mapping_status": "test_exercise_proxy_no_topic_id",
        })
    summary = pd.DataFrame(rows)
    if published_kpi is not None and not published_kpi.empty:
        available = [col for col in ["user_id", "test_id", "avg_accuracy_safe", "marks"] if col in published_kpi.columns]
        summary = summary.merge(published_kpi[available].rename(columns={"avg_accuracy_safe": "published_accuracy_safe", "marks": "published_marks"}), on=["user_id", "test_id"], how="left")
    return summary

# published_kpi = build_published_kpi_dataset(dq_attempts)
# proxy_sequence = build_proxy_sequence_dataset(dq_attempts)
# user_test_summary = build_user_test_readiness_summary(proxy_sequence, published_kpi)
# published_kpi.to_csv(EXPORT_DIR / "published_kpi_user_test.csv", index=False)
# proxy_sequence.to_csv(EXPORT_DIR / "proxy_sequence_attempts.csv", index=False)
# user_test_summary.to_csv(EXPORT_DIR / "v13_user_test_readiness_summary.csv", index=False)

## Group / Cohort Summary

CAS remains a proxy in v1.3. If true class membership is not confirmed, label cohort outputs as inferred.

In [ ]:
def build_group_readiness_summary(user_test_summary):
    group_frames = []
    for group_level, group_col in [("class", "class_id"), ("institute", "institute"), ("city", "city"), ("country", "country"), ("institution_or_subscriber", "subscriber_id"), ("test", "test_id")]:
        if group_col not in user_test_summary.columns:
            continue
        group_keys = [group_col] if group_col == "test_id" else [group_col, "test_id"]
        grouped = user_test_summary.groupby(group_keys, dropna=False).agg(learner_count=("user_id", "nunique"), repeated_group_count=("attempt_count", lambda s: int((s >= 2).sum())), mean_bls_score_pct=("bls_score_pct", "mean"), mean_current_als_score_pct=("current_als_score_pct", "mean"), mean_potential_als_score_pct=("potential_als_score_pct", "mean"), mean_learning_gain_pct=("learning_gain_pct", "mean")).reset_index()
        if group_col == "test_id":
            grouped["group_id"] = grouped["test_id"]
        else:
            grouped = grouped.rename(columns={group_col: "group_id"})
        grouped["group_level"] = group_level
        grouped["cas_proxy_score_pct"] = grouped["mean_current_als_score_pct"]
        grouped["caveat_note"] = "CAS proxy from test/exercise attempts; not true class ALS calibration."
        group_frames.append(grouped)
    return pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()

# group_summary = build_group_readiness_summary(user_test_summary)
# group_summary.to_csv(EXPORT_DIR / "v13_group_readiness_summary.csv", index=False)

## Smoke Checks

Run these before treating exports as dashboard-ready.

In [ ]:
def smoke_report(raw_attempts, dq_attempts, published_kpi, proxy_sequence, user_test_summary):
    repeated_groups = proxy_sequence.groupby(["user_id", "test_id"], dropna=False).size()
    report = {
        "raw_rows": len(raw_attempts), "dq_annotated_rows": len(dq_attempts), "published_kpi_rows": len(published_kpi), "proxy_sequence_rows": len(proxy_sequence), "user_test_summary_rows": len(user_test_summary),
        "repeated_user_test_groups": int((repeated_groups >= 2).sum()),
        "inactive_zero_attempt_rows": int(dq_attempts.get("inactive_zero_attempt", pd.Series(dtype=bool)).sum()),
        "total_marks_suspect_rows": int(dq_attempts.get("total_marks_suspect", pd.Series(dtype=bool)).sum()),
        "total_questions_suspect_rows": int(dq_attempts.get("total_questions_suspect", pd.Series(dtype=bool)).sum()),
        "denominator_conflict_rows": int(dq_attempts.get("denominator_conflict_flag", pd.Series(dtype=bool)).sum()),
        "max_marks_db_bank_count_rows": int(dq_attempts.get("max_marks_db_is_bank_count", pd.Series(dtype=bool)).sum()),
        "denominator_source_counts": dq_attempts.get("accuracy_denominator_source", pd.Series(dtype=object)).value_counts(dropna=False).to_dict(),
        "missing_finished_at_column": bool(dq_attempts.get("missing_finished_at_column", pd.Series([False])).any()),
        "accuracy_min": float(dq_attempts["avg_accuracy_safe"].min(skipna=True)) if "avg_accuracy_safe" in dq_attempts else None,
        "accuracy_max": float(dq_attempts["avg_accuracy_safe"].max(skipna=True)) if "avg_accuracy_safe" in dq_attempts else None,
        "learning_gain_min": float(user_test_summary["learning_gain_pct"].min(skipna=True)) if "learning_gain_pct" in user_test_summary else None,
        "learning_gain_max": float(user_test_summary["learning_gain_pct"].max(skipna=True)) if "learning_gain_pct" in user_test_summary else None,
    }
    return pd.DataFrame([report])

def write_manifest(export_dir, report_df, extra_notes=None):
    manifest = {"run_label": RUN_LABEL, "generated_at_utc": pd.Timestamp.utcnow().isoformat(), "v1_3_scope": "Test / Exercise Readiness proxy dataset", "topic_level_status": "Deferred to v2; no canonical topic_id in current source.", "subject_level_status": "May be inferred from test_name only; not canonical.", "class_level_status": "Use class_id with membership validation when available.", "dq_note": "Raw rows are annotated before filtering. Published KPI and proxy-sequence outputs are separate.", "smoke_report": report_df.to_dict(orient="records")[0] if report_df is not None and not report_df.empty else {}, "extra_notes": extra_notes or []}
    path = Path(export_dir) / "manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return path

# report = smoke_report(raw_attempts, dq_attempts, published_kpi, proxy_sequence, user_test_summary)
# display(report.T)
# report.to_csv(EXPORT_DIR / "smoke_report.csv", index=False)
# manifest_path = write_manifest(EXPORT_DIR, report)